In [332]:
# Import the required libraries
import yaml
import joblib
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

# **1. Load & Update Configuration**

In [333]:
def load_config(path_config):
    """
    Load the configuration file (config.yaml).

    Parameters:
    ----------
    path_config : str
        Configuration file location.

    Returns:
    -------
    params : dict
        The configuration parameters.
    """

    # try to load config.yaml file
    try:
        with open(path_config, 'r') as file:
            config = yaml.safe_load(file)
    except FileNotFoundError:
        raise FileNotFoundError(f'Configuration file is not found in {path_config}.')
    except yaml.YAMLError as e:
        raise ValueError('Invalid yaml format') from e
    
    return config

In [334]:
# Function to update configuration parameter.
def update_config(key, value, config, path_config):
    """
    Update the configuration parameter values.

    Parameters:
    ----------
    key : str
        The key to be updated.

    value : any type supported in Python
        The updated value.

    config : dict
        Loaded configuration parameters.

    path_config : str
        Configuration file location.

    Returns:
    -------
    config : dict
        Updated configuration parameters.
    """

    # To maintain the raw config immutable.
    config = config.copy()

    # Update the configuration parameters.
    config[key] = value

    with open(path_config, 'w') as file:
        yaml.dump(config, file)

    print(f"Config Updated! \nKey: {key} \nValue: {value}\n")

    # Reload the updated configuration parameters.
    config = load_config(path_config)

    return config

In [335]:
# load config
config_path = '../config/config.yaml'
config = load_config(path_config=config_path)
config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'impute_co': 11.0,
 'impute_no2': 18.0,
 'impute_o3': 29.0,
 'impute_pm10': {'BAIK': 28.683098591549296, 'TIDAK BAIK': 55.53523357086303},
 'impute_pm25': {'BAIK': 39.472727272727276, 'TIDAK BAIK': 82.82560879811469},
 'impute_so2': 35.01016702977487,
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_ohe_stasiun': '../models/ohe_stasiun.pkl',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl

# **2. Load Data**

In [336]:
def load_data(config):
    """
    Load every set of data.

    Parameters:
    ----------
    config : dict
        The loaded configuration file.

    Returns:
    -------
    data_train, data_valid, data_test : pd.DataFrame
        The loaded data.
    """
    # Load the train set.
    X_train = joblib.load(config["path_train_set"][0])
    y_train = joblib.load(config["path_train_set"][1])

    # Load the valid set.
    X_valid = joblib.load(config["path_valid_set"][0])
    y_valid = joblib.load(config["path_valid_set"][1])

    # Load the test set.
    X_test = joblib.load(config["path_test_set"][0])
    y_test = joblib.load(config["path_test_set"][1])

    # Concatenate the X and y of each set.
    data_train = pd.concat([X_train, y_train], axis=1)
    data_valid = pd.concat([X_valid, y_valid], axis=1)
    data_test = pd.concat([X_test, y_test], axis=1)

    # Validate the proportion.
    num_all_data = int(data_train.shape[0]) + int(data_valid.shape[0]) + int(data_test.shape[0])
    print(f"Data train proportion : {len(X_train) / num_all_data}")
    print(f"Data valid proportion : {len(X_valid) / num_all_data}")
    print(f"Data test proportion  : {len(X_test) / num_all_data}")

    return data_train, data_valid, data_test

In [337]:
data_train, data_valid, data_test = load_data(config=config)

Data train proportion : 0.7997793712079426
Data valid proportion : 0.09983452840595698
Data test proportion  : 0.10038610038610038


# **3. Join Categories**

In [338]:
# make a defined function to join categories
label = config['label']

def join_categories(data, config):

    # check if label found in dataset
    if config['label'] in data.columns.tolist():
        data = data.copy()

        # rename 'SEDANG' to 'TIDAK SEHAT'
        data[label] = data[label].replace('SEDANG', 'TIDAK SEHAT')
        data[label] = data[label].replace('TIDAK SEHAT', 'TIDAK BAIK')
        
        return data

    else:
        raise RuntimeError('Label is not detected in the dataset.')

In [339]:
# Update the configuration parameter
config = update_config(
    key='label_categories_new',
    value=['BAIK', 'TIDAK BAIK'],
    config=config,
    path_config=config_path
)

Config Updated! 
Key: label_categories_new 
Value: ['BAIK', 'TIDAK BAIK']



In [340]:
# sanity check on updated config
config

{'co_range': [-1, 47],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'impute_co': 11.0,
 'impute_no2': 18.0,
 'impute_o3': 29.0,
 'impute_pm10': {'BAIK': 28.683098591549296, 'TIDAK BAIK': 55.53523357086303},
 'impute_pm25': {'BAIK': 39.472727272727276, 'TIDAK BAIK': 82.82560879811469},
 'impute_so2': 35.01016702977487,
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'no2_range': [-1, 65],
 'o3_range': [-1, 151],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_ohe_stasiun': '../models/ohe_stasiun.pkl',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl

In [341]:
# join categories on each data
data_train = join_categories(data_train, config)
data_valid = join_categories(data_valid, config)
data_test  = join_categories(data_test, config)

data_name = ['data_train', 'data_valid', 'data_test']
data_list = [data_train, data_valid, data_test]
i = 0

for data in data_list:
    print(f'Data: {data_name[i]}')
    print(data[label].value_counts(), '\n')
    i += 1

Data: data_train
category
TIDAK BAIK    1299
BAIK           151
Name: count, dtype: int64 

Data: data_valid
category
TIDAK BAIK    162
BAIK           19
Name: count, dtype: int64 

Data: data_test
category
TIDAK BAIK    163
BAIK           19
Name: count, dtype: int64 



# **4. Handling Missing Value**
----

1. Convert all -1 values into NaN
2. Impute the missing values for all columns

In [342]:
# make a defined function to replace -1 with NaN
def replace_nan(data):
    data = data.copy()
    data = data.replace(-1, np.nan)
    return data

In [343]:
# join categories on each data
data_train = replace_nan(data=data_train)
data_valid = replace_nan(data=data_valid)
data_test  = replace_nan(data=data_test)

data_name = ['data_train', 'data_valid', 'data_test']
data_list = [data_train, data_valid, data_test]
i = 0

for data in data_list:
    print(f'Data: {data_name[i]}')
    print(data.isnull().sum(), '\n')
    i += 1

Data: data_train
stasiun      0
pm10        45
pm25        67
so2         73
co          12
o3          38
no2         17
category     0
dtype: int64 

Data: data_valid
stasiun      0
pm10         2
pm25        12
so2          6
co           2
o3           7
no2          2
category     0
dtype: int64 

Data: data_test
stasiun      0
pm10         6
pm25         7
so2         18
co           2
o3           3
no2          0
category     0
dtype: int64 



## **4.1 pm10 Imputation**

In [344]:
# imputation using class mean
pm10_impute_baik = float(data_train[data_train[label] == 'BAIK']['pm10'].mean())
pm10_impute_tidak_baik = float(data_train[data_train[label] == 'TIDAK BAIK']['pm10'].mean())

print(f"Mean pm10 class BAIK       : {pm10_impute_baik}")
print(f"Mean pm10 class TIDAK BAIK : {pm10_impute_tidak_baik}")

Mean pm10 class BAIK       : 28.683098591549296
Mean pm10 class TIDAK BAIK : 55.53523357086303


In [345]:
# update the configuration parameter
config = update_config(
    key='impute_pm10',
    value={'BAIK': pm10_impute_baik,
           'TIDAK BAIK': pm10_impute_tidak_baik},
    config=config,
    path_config=config_path
)

Config Updated! 
Key: impute_pm10 
Value: {'BAIK': 28.683098591549296, 'TIDAK BAIK': 55.53523357086303}



In [346]:
# impute the missing values on data train
data_train.loc[data_train[(data_train.category == 'BAIK') & data_train.pm10.isnull() == True].index, 'pm10'] = pm10_impute_baik

data_train.loc[data_train[(data_train.category == 'TIDAK BAIK') & data_train.pm10.isnull() == True].index, 'pm10'] = pm10_impute_tidak_baik

print(f"Num of missing value pm10 class BAIK       : {data_train[data_train['category'] == 'BAIK']['pm10'].isnull().sum()}")
print(f"Num of missing value pm10 class TIDAK BAIK : {data_train[data_train['category'] == 'TIDAK BAIK']['pm10'].isnull().sum()}")

Num of missing value pm10 class BAIK       : 0
Num of missing value pm10 class TIDAK BAIK : 0


In [347]:
# impute the missing values on data valid
data_valid.loc[data_valid[(data_valid.category == 'BAIK') & data_valid.pm10.isnull() == True].index, 'pm10'] = pm10_impute_baik

data_valid.loc[data_valid[(data_valid.category == 'TIDAK BAIK') & data_valid.pm10.isnull() == True].index, 'pm10'] = pm10_impute_tidak_baik

print(f"Num of missing value pm10 class BAIK       : {data_valid[data_valid['category'] == 'BAIK']['pm10'].isnull().sum()}")
print(f"Num of missing value pm10 class TIDAK BAIK : {data_valid[data_valid['category'] == 'TIDAK BAIK']['pm10'].isnull().sum()}")

Num of missing value pm10 class BAIK       : 0
Num of missing value pm10 class TIDAK BAIK : 0


In [348]:
# impute the missing values on data test
data_test.loc[data_test[(data_test.category == 'BAIK') & data_test.pm10.isnull() == True].index, 'pm10'] = pm10_impute_baik

data_test.loc[data_test[(data_test.category == 'TIDAK BAIK') & data_test.pm10.isnull() == True].index, 'pm10'] = pm10_impute_tidak_baik

print(f"Num of missing value pm10 class BAIK       : {data_test[data_test['category'] == 'BAIK']['pm10'].isnull().sum()}")
print(f"Num of missing value pm10 class TIDAK BAIK : {data_test[data_test['category'] == 'TIDAK BAIK']['pm10'].isnull().sum()}")

Num of missing value pm10 class BAIK       : 0
Num of missing value pm10 class TIDAK BAIK : 0


## **4.2 pm25 Imputation**

In [349]:
# imputation using class mean
pm25_impute_baik = float(data_train[data_train[label] == 'BAIK']['pm25'].mean())
pm25_impute_tidak_baik = float(data_train[data_train[label] == 'TIDAK BAIK']['pm25'].mean())

print(f"Mean pm25 class BAIK       : {pm25_impute_baik}")
print(f"Mean pm25 class TIDAK BAIK : {pm25_impute_tidak_baik}")

Mean pm25 class BAIK       : 39.472727272727276
Mean pm25 class TIDAK BAIK : 82.82560879811469


In [350]:
# update the configuration parameter
config = update_config(
    key='impute_pm25',
    value={'BAIK': pm25_impute_baik,
           'TIDAK BAIK': pm25_impute_tidak_baik},
    config=config,
    path_config=config_path
)

Config Updated! 
Key: impute_pm25 
Value: {'BAIK': 39.472727272727276, 'TIDAK BAIK': 82.82560879811469}



In [351]:
# impute the missing values on data train
data_train.loc[data_train[(data_train.category == 'BAIK') & data_train.pm25.isnull() == True].index, 'pm25'] = pm25_impute_baik

data_train.loc[data_train[(data_train.category == 'TIDAK BAIK') & data_train.pm25.isnull() == True].index, 'pm25'] = pm25_impute_tidak_baik

print(f"Num of missing value pm25 class BAIK       : {data_train[data_train['category'] == 'BAIK']['pm25'].isnull().sum()}")
print(f"Num of missing value pm25 class TIDAK BAIK : {data_train[data_train['category'] == 'TIDAK BAIK']['pm25'].isnull().sum()}")

Num of missing value pm25 class BAIK       : 0
Num of missing value pm25 class TIDAK BAIK : 0


In [352]:
# impute the missing values on data valid
data_valid.loc[data_valid[(data_valid.category == 'BAIK') & data_valid.pm25.isnull() == True].index, 'pm25'] = pm25_impute_baik

data_valid.loc[data_valid[(data_valid.category == 'TIDAK BAIK') & data_valid.pm25.isnull() == True].index, 'pm25'] = pm25_impute_tidak_baik

print(f"Num of missing value pm25 class BAIK       : {data_valid[data_valid['category'] == 'BAIK']['pm25'].isnull().sum()}")
print(f"Num of missing value pm25 class TIDAK BAIK : {data_valid[data_valid['category'] == 'TIDAK BAIK']['pm25'].isnull().sum()}")

Num of missing value pm25 class BAIK       : 0
Num of missing value pm25 class TIDAK BAIK : 0


In [353]:
# impute the missing values on data test
data_test.loc[data_test[(data_test.category == 'BAIK') & data_test.pm25.isnull() == True].index, 'pm25'] = pm25_impute_baik

data_test.loc[data_test[(data_test.category == 'TIDAK BAIK') & data_test.pm25.isnull() == True].index, 'pm25'] = pm25_impute_tidak_baik

print(f"Num of missing value pm25 class BAIK       : {data_test[data_test['category'] == 'BAIK']['pm25'].isnull().sum()}")
print(f"Num of missing value pm25 class TIDAK BAIK : {data_test[data_test['category'] == 'TIDAK BAIK']['pm25'].isnull().sum()}")

Num of missing value pm25 class BAIK       : 0
Num of missing value pm25 class TIDAK BAIK : 0


## **4.3 so2, co, o3, no2 Imputation**

In [354]:
# so2 imputed using the mean
# co, o3, no2 imputed using the median

impute_so2 = float(data_train["so2"].mean())
impute_co = float(data_train["co"].median())
impute_o3 = float(data_train["o3"].median())
impute_no2 = float(data_train["no2"].median())

impute_values = {
    'so2' : impute_so2,
    'co' : impute_co,
    'o3' : impute_o3,
    'no2' : impute_no2
}

impute_values

{'so2': 35.01016702977487, 'co': 11.0, 'o3': 29.0, 'no2': 18.0}

In [355]:
# Update the configuration parameter.
cols = ['so2', 'co', 'o3', 'no2']
param_keys = ['impute_so2', 'impute_co', 'impute_o3', 'impute_no2']

for col, param_key in zip(cols, param_keys):
    config = update_config(
        key = param_key,
        value = impute_values[col],
        config = config,
        path_config = config_path
    )

Config Updated! 
Key: impute_so2 
Value: 35.01016702977487

Config Updated! 
Key: impute_co 
Value: 11.0

Config Updated! 
Key: impute_o3 
Value: 29.0

Config Updated! 
Key: impute_no2 
Value: 18.0



In [356]:
# impute the missing values
data_train = data_train.fillna(value=impute_values)
data_valid = data_valid.fillna(value=impute_values)
data_test = data_test.fillna(value=impute_values)

data_name = ['data_train', 'data_valid', 'data_test']
data_list = [data_train, data_valid, data_test]
i = 0

for data in data_list:
    print(f'Data: {data_name[i]}')
    print(data.isnull().sum(), '\n')
    i += 1

Data: data_train
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64 

Data: data_valid
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64 

Data: data_test
stasiun     0
pm10        0
pm25        0
so2         0
co          0
o3          0
no2         0
category    0
dtype: int64 



# **5. Encoding Stasiun**

In [357]:
def fit_ohe_encoder(X_stasiun):
    '''
    Fit the OHE Encoder

    Parameters:
    ----------
    X_stasiun : pd.DataFrame
        Categorical input data
    
    Returns:
    -------
    ohe_encoder : sklearn object
        Fitted OHE encoder object
    '''
    
    ohe_encoder = OneHotEncoder(sparse_output=False)
    ohe_encoder.fit(np.array(X_stasiun).reshape(-1, 1)) # series to df

    # serialize the ohe_encoder object
    joblib.dump(ohe_encoder, '../models/ohe_stasiun.pkl')
    
    return ohe_encoder

def transform_ohe_encoder(dataset, transformed_column, ohe_path):
    """
    Transform the categorical input data using OHE encoder
    
    Parameters:
    ----------
    dataset : pd.DataFrame
        Data to be transformed.
        
    transformed_column : str
        The column name.
        
    ohe_path : str
        The path to the ohe_encoder object.
        
    Returns:
    -------
    dataset : pd.DataFrame
        The concatenated set data with OHE columns.
    """    

    dataset = dataset.copy()

    # load the ohe_encoder
    ohe_encoder = joblib.load(ohe_path)

    # transform the data
    X_stasiun = np.array(dataset[transformed_column]).reshape(-1,1)
    stasiun_features = ohe_encoder.transform(X_stasiun)

    # convert to dataframe
    stasiun_features = pd.DataFrame(data=stasiun_features.tolist(),
                                    columns=list(ohe_encoder.categories_[0]))
    
    # set index by original set data index
    stasiun_features.set_index(dataset.index, inplace=True)

    # concat the new features with the original set data
    dataset = pd.concat([dataset, stasiun_features], axis=1)

    # drop the stasiun column
    dataset.drop(columns='stasiun', inplace=True)

    # convert columns type to string
    new_col = [str(col_name) for col_name in dataset.columns.tolist()]
    dataset.columns = new_col

    # return the feature engineered data
    return dataset

In [358]:
ohe_stasiun = fit_ohe_encoder(config['range_stasiun'])

In [359]:
# update the config parameter
PATH_OHE_STASIUN = '../models/ohe_stasiun.pkl'

config = update_config(
    key='path_ohe_stasiun',
    value=PATH_OHE_STASIUN,
    config=config,
    path_config=config_path
)

Config Updated! 
Key: path_ohe_stasiun 
Value: ../models/ohe_stasiun.pkl



In [360]:
# encode the stasiun column in train, valid, test
data_train = transform_ohe_encoder(
    dataset=data_train,
    transformed_column='stasiun',
    ohe_path=config['path_ohe_stasiun']
)

data_valid = transform_ohe_encoder(
    dataset=data_valid,
    transformed_column='stasiun',
    ohe_path=config['path_ohe_stasiun']
)

data_test = transform_ohe_encoder(
    dataset=data_test,
    transformed_column='stasiun',
    ohe_path=config['path_ohe_stasiun']
)

In [361]:
# sanity check
data_train.head()

,pm10,pm25,so2,co,o3,no2,category,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat
224,50.0,76.000000,20.0,8.0,71.0,15.0,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0
1712,33.0,58.000000,42.0,11.0,43.0,19.0,TIDAK BAIK,0.0,1.0,0.0,0.0,0.0
1028,51.0,78.000000,37.0,12.0,23.0,19.0,TIDAK BAIK,0.0,0.0,0.0,1.0,0.0
391,46.0,39.472727,17.0,17.0,41.0,10.0,BAIK,0.0,0.0,1.0,0.0,0.0
1547,68.0,102.000000,31.0,13.0,20.0,30.0,TIDAK BAIK,1.0,0.0,0.0,0.0,0.0


In [362]:
data_valid.head()

,pm10,pm25,so2,co,o3,no2,category,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat
82,81.0,118.0,27.0,15.0,37.0,16.0,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0
1150,68.0,118.0,40.0,16.0,26.0,27.0,TIDAK BAIK,0.0,0.0,0.0,1.0,0.0
549,63.0,83.0,42.0,12.0,27.0,11.0,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0
850,27.0,54.0,25.0,6.0,24.0,7.0,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0
388,58.0,86.0,22.0,11.0,43.0,9.0,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0


In [363]:
data_test.head()

,pm10,pm25,so2,co,o3,no2,category,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat
1317,63.0,114.0,39.000000,11.0,15.0,21,TIDAK BAIK,0.0,0.0,0.0,1.0,0.0
1648,66.0,115.0,35.000000,11.0,36.0,20,TIDAK BAIK,0.0,0.0,0.0,0.0,1.0
803,30.0,38.0,37.000000,11.0,57.0,9,TIDAK BAIK,0.0,1.0,0.0,0.0,0.0
1125,73.0,108.0,48.000000,13.0,29.0,15,TIDAK BAIK,0.0,0.0,1.0,0.0,0.0
1503,70.0,107.0,35.010167,11.0,39.0,16,TIDAK BAIK,0.0,0.0,0.0,0.0,1.0


# **6. Scaling Data**
----

In [364]:
def fit_scaler(X_concat):
    """
    Fit the scaler
    
    Parameters:
    ----------
    X_concat : pd.DataFrame
        Input data (all features must be in numeric form)
        
    Returns:
    -------
    scaler : sklearn object
        Fitted scaler object (storing the mean & std of all features)
    """
    
    scaler = StandardScaler()
    scaler.fit(X_concat)

    # Serialize the ohe_encoder object.    
    joblib.dump(scaler, '../models/scaler.pkl')
    
    return scaler

def transform_scaler(X_concat, scaler):
    """
    Transform the data using scaler
    
    Parameters:
    ----------
    X_concat : pd.DataFrame
        Input data (all features must be in numeric form)
        
    scaler : sklearn object
        Fitted scaler object (storing the mean & std of all features)
        
    Returns:
    -------
    X_concat_scaled : pd.DataFrame
        Scaled data
    """
    
    X_concat = X_concat.copy()
    
    # Transform the data
    X_concat_scaled = pd.DataFrame(
        scaler.transform(X_concat),
        columns = X_concat.columns,
        index = X_concat.index
    )
    
    return X_concat_scaled

In [365]:
# fit transform data train
target = data_train['category']

# Fit the scaler
scaler = fit_scaler(data_train.drop(columns='category'))

# Transform the data
data_train = transform_scaler(X_concat = data_train.drop(columns='category'),
                              scaler = scaler)

data_train = pd.concat([data_train, target], axis=1)

data_train.head()

,pm10,pm25,so2,co,o3,no2,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,category
224,-0.186993,-0.092878,-1.237096,-0.759346,2.737990,-0.478647,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK
1712,-1.347628,-0.816314,0.576082,-0.137985,0.776381,-0.031817,-0.510754,2.013024,-0.496765,-0.494606,-0.501077,TIDAK BAIK
1028,-0.118720,-0.012497,0.163996,0.069135,-0.624768,-0.031817,-0.510754,-0.496765,-0.496765,2.021811,-0.501077,TIDAK BAIK
391,-0.460083,-1.560941,-1.484347,1.104737,0.636267,-1.037184,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,BAIK
1547,1.041916,0.952084,-0.330507,0.276256,-0.834940,1.196965,1.957890,-0.496765,-0.496765,-0.494606,-0.501077,TIDAK BAIK


In [366]:
# transform data valid
target = data_valid['category']

# Transform the data
data_valid = transform_scaler(X_concat = data_valid.drop(columns='category'),
                              scaler = scaler)

data_valid = pd.concat([data_valid, target], axis=1)

data_valid.head()

,pm10,pm25,so2,co,o3,no2,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,category
82,1.929461,1.595138,-0.660175,0.690496,0.356037,-0.366940,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK
1150,1.041916,1.595138,0.411248,0.897617,-0.414595,0.861842,-0.510754,-0.496765,-0.496765,2.021811,-0.501077,TIDAK BAIK
549,0.700552,0.188458,0.576082,0.069135,-0.344538,-0.925477,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK
850,-1.757264,-0.977077,-0.825010,-1.173587,-0.554710,-1.372307,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK
388,0.359189,0.309030,-1.072261,-0.137985,0.776381,-1.148892,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK


In [367]:
# transform data test
target = data_test['category']

# Transform the data
data_test = transform_scaler(X_concat = data_test.drop(columns='category'),
                              scaler = scaler)

data_test = pd.concat([data_test, target], axis=1)

data_test.head()

,pm10,pm25,so2,co,o3,no2,DKI1 (Bunderan HI),DKI2 (Kelapa Gading),DKI3 (Jagakarsa),DKI4 (Lubang Buaya),DKI5 (Kebon Jeruk) Jakarta Barat,category
1317,0.700552,1.434375,0.328831,-0.137985,-1.185227,0.191598,-0.510754,-0.496765,-0.496765,2.021811,-0.501077,TIDAK BAIK
1648,0.905370,1.474566,-0.000838,-0.137985,0.285979,0.079890,-0.510754,-0.496765,-0.496765,-0.494606,1.995700,TIDAK BAIK
803,-1.552446,-1.620131,0.163996,-0.137985,1.757186,-1.148892,-0.510754,2.013024,-0.496765,-0.494606,-0.501077,TIDAK BAIK
1125,1.383279,1.193230,1.070585,0.276256,-0.204423,-0.478647,-0.510754,-0.496765,2.013024,-0.494606,-0.501077,TIDAK BAIK
1503,1.178461,1.153039,0.000000,-0.137985,0.496152,-0.366940,-0.510754,-0.496765,-0.496765,-0.494606,1.995700,TIDAK BAIK


# **7. Balancing Label**
----

In [368]:
data_train['category'].value_counts(normalize=True)

category
TIDAK BAIK    0.895862
BAIK          0.104138
Name: proportion, dtype: float64

In [369]:
# Undersampling.
rus = RandomUnderSampler(random_state = 123)

X_rus, y_rus = rus.fit_resample(data_train.drop('category', axis=1),
                                data_train['category'])

data_train_rus = pd.concat([X_rus, y_rus], axis=1)

In [370]:
# Oversampling.
ros = RandomOverSampler(random_state = 123)

X_ros, y_ros = ros.fit_resample(data_train.drop('category', axis=1),
                                data_train['category'])

data_train_ros = pd.concat([X_ros, y_ros], axis=1)

In [371]:
# SMOTE.
smote = SMOTE(random_state = 123)

X_sm, y_sm = smote.fit_resample(data_train.drop('category', axis=1),
                                data_train['category'])

data_train_sm = pd.concat([X_sm, y_sm], axis=1)

# **8. Label Encoding**
----

In [372]:
def fit_le_encoder(y_categori):
    """
    Fit the LE encoder
    
    Parameters:
    ----------
    y_categori : pd.Series
        Categorical input label
        
    Returns:
    -------
    le_encoder : sklearn object
        Fitted LE encoder object
    """
    
    le_encoder = LabelEncoder()
    le_encoder.fit(y_categori)

    # Serialize the ohe_encoder object.    
    joblib.dump(le_encoder, '../models/le_encoder.pkl')
    
    return le_encoder

def transform_le_encoder(y_categori, le_encoder):
    """
    Transform the categorical input label using LE encoder
    
    Parameters:
    ----------
    y_categori : pd.Series
        Categorical input label
        
    le_encoder : sklearn object
        Fitted LE encoder object
        
    Returns:
    -------
    y_categori_encoded : pd.DataFrame
        Encoded categorical input label
    """
    
    y_categori = y_categori.copy()
    
    # Transform the data
    y_categori_encoded = pd.Series(
        le_encoder.transform(y_categori),        
    )
    
    return y_categori_encoded

In [373]:
# Fit the label encoder.
le_category = fit_le_encoder(config["label_categories_new"])

In [374]:
# Transform RUS data.
y_rus = transform_le_encoder(y_rus, le_category)

In [375]:
# Transform ROS data.
y_ros = transform_le_encoder(y_ros, le_category)

In [376]:
# Transform SMOTE data.
y_sm = transform_le_encoder(y_sm, le_category)

In [377]:
# Transform valid set.
category_encoded = transform_le_encoder(data_valid["category"], le_category)

data_valid["category"] = category_encoded.values.tolist()

In [378]:
# Transform test set.
category_encoded = transform_le_encoder(data_test["category"], le_category)

data_test["category"] = category_encoded.values.tolist()

# **9. Data Serialization**
----

In [379]:
# undersampling train
joblib.dump(X_rus, "../data/processed/X_rus.pkl")
joblib.dump(y_rus, "../data/processed/y_rus.pkl")

# oversampling train
joblib.dump(X_ros, "../data/processed/X_ros.pkl")
joblib.dump(y_ros, "../data/processed/y_ros.pkl")

# smote train
joblib.dump(X_sm, "../data/processed/X_sm.pkl")
joblib.dump(y_sm, "../data/processed/y_sm.pkl")

# data valid
joblib.dump(data_valid.drop(columns='category'), "../data/processed/X_valid_feng.pkl")
joblib.dump(data_valid['category'], "../data/processed/y_valid_feng.pkl")

# data test
joblib.dump(data_test.drop(columns='category'), "../data/processed/X_test_feng.pkl")
joblib.dump(data_test['category'], "../data/processed/y_test_feng.pkl")

['../data/processed/y_test_feng.pkl']